## 🧠 What is LangChain?

**LangChain** is a framework to build applications with **Large Language Models (LLMs), designed to help developers build applications that integrate LLMs (like GPT-4, Claude, LLaMA, Qwen, etc.) with other data sources, tools, and systems in a structured and modular way.

It is providing:
* **Chains** to combine LLMs, prompts, memory, tools, etc.
* **Agents** for tool-using LLMs
* **Retrievers** + **VectorStores** for RAG
* Support for **evaluation**, **memory**, **tracing**, and more

In simple terms — it helps you build advanced AI applications that do more than just chat — things like:

* Accessing and using **external data** (PDFs, websites, databases, APIs)
* Keeping **memory** of conversations
* Running **multi-step workflows** or chains of reasoning
* Calling **external tools** or functions (retrieval, search, calculators, APIs)
* Integrating with **retrieval systems** (RAG — Retrieval-Augmented Generation)
* Managing **agents** that decide what actions to take

---

### Core Concepts

| Concept             | Meaning                                                                                               |
| ------------------- | ----------------------------------------------------------------------------------------------------- |
| **Chain**           | A sequence of calls (LLM calls + other logic). Example: "Get query → search DB → summarize → answer." |
| **Prompt Template** | Dynamic creation of prompts to LLMs (with variables).                                                 |
| **LLM Wrappers**    | Connect to LLMs (OpenAI, Hugging Face, Anthropic, local models...).                                   |
| **Memory**          | Store conversation history (to enable context-aware interactions).                                    |
| **Retrievers**      | Components to fetch relevant data from sources (DBs, vector stores, documents).                       |
| **Agents**          | LLMs that can choose what tool to use next — dynamic decision-making.                                 |
| **Tools**           | External APIs, code functions, calculators that the agent can invoke.                                 |

---

### Example Use Cases

* **Chatbots** with memory & context
* **RAG systems**: Ask questions on your private docs
* **Automated data extraction**
*  **Assistants** that can search the web, run tools
*  **Multi-step workflows** (reasoning, planning)

---

### Why Use LangChain?

* Saves you from reinventing the wheel when building **AI apps**
* Modular — you can pick just what you need
* Works with **OpenAI**, **HuggingFace**, **LLama.cpp**, **Qwen**, **Groq**, **Anthropic**...
* Lots of **integrations**: PDF loaders, vector stores (FAISS, Pinecone, Chroma), databases, APIs
* Great community & ecosystem

---

If you tell me your use case (for example: "I want to extract structured data from PDFs" or "I want to build a chatbot on my website"), I can give you more targeted examples of how LangChain helps 🚀. Want me to?


## 📍 Explanation of LangChain Setting

| Variable               | Purpose                                                              |
| ---------------------- | -------------------------------------------------------------------- |
| `LANGCHAIN_API_KEY`    | 🔑 Authenticates you to LangSmith (must start with `lsv2_...`).      |
| `LANGCHAIN_ENDPOINT`   | 🌐 URL for LangSmith API (always `https://api.smith.langchain.com`). |
| `LANGCHAIN_TRACING_V2` | ✅ Enables **LangSmith v2 tracing** (true/false).                     |
| `LANGCHAIN_PROJECT`    | 📁 Optional, groups your runs under a project name in the dashboard. |


In [ ]:
import torch
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pkg_resources")

print(torch.version.cuda)  # type: ignore
print(torch.cuda.is_available())
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA not available")

### Load Groq API KEY

In [ ]:
import os

# Import load_dotenv function from python-dotenv package
from dotenv import load_dotenv

# Load environment variables from a .env file into the environment
load_dotenv()
print(os.getenv("GROQ_API_KEY"))  # Should print your key (or part of it)

### Part 1: Overview of RAG Basic Pipeline

In [ ]:
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_groq import ChatGroq

# from langchain_community.embeddings import HuggingFaceEmbeddings  # Instead of OpenAIEmbeddings if you want to use HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings  # ✅ Updated

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cuda"}) # Use HuggingFaceEmbeddings instead of OpenAIEmbeddings

# Embed
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)

retriever = vectorstore.as_retriever()

#### RETRIEVAL and GENERATION ####

# Prompt
prompt = hub.pull("rlm/rag-prompt")

# LLM
# llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
llm = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0.0,
    max_tokens=1024,
)
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser(  # type: ignore
        output_key="answer",
        input_key="question",
        output_variable_name="answer",
    )

)

# Question
ans = rag_chain.invoke("What is this topic talking about?")


* **Chroma:**
  - >This refers to a vector store implementation called Chroma, which is used to store and index document embeddings efficiently. A vector store is a database optimized for searching vectors, typically used for similarity search in embeddings.

In [ ]:
ans

Stages of the RAG App:
1. **Indexing**: Load and split documents, then embed them into a vector store.
2. **Retrieval and Generation**: Use a retriever to fetch relevant documents based on a query, then generate an answer using a language model and a prompt template.
3. *Generation*: The final answer is generated by the language model based on the retrieved context and the question asked.


### STAGE 1: Indexing

In [ ]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

* Tokenizations:
- "[Count tokens](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb) considering [~4 char / token](https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them)"

* `cl100k_base` is the **tokenizer encoding name** used by many OpenAI models, such as `gpt-3.5-turbo` and `gpt-4`.

---

### 🔍 What is it exactly?

* `cl100k_base` is a **token encoding scheme** designed for OpenAI models that use **Byte-Pair Encoding (BPE)**.
* It maps strings to tokens in a way that's optimized for both performance and compatibility with OpenAI's chat models.
* This encoding determines **how many tokens** a given string will consume in models like:

  * `gpt-3.5-turbo`
  * `gpt-4`
  * `text-embedding-ada-002`
  * and other newer OpenAI APIs.

---

### 🧠 Why does it matter?

LLMs (like OpenAI's models) have **token limits**, so knowing how many tokens you're using helps avoid:

* Going over limits (e.g. 4096 or 8192 tokens),
* Managing cost,
* Efficiently chunking text (e.g. for embedding or retrieval).

### 🔄 Common alternatives to `cl100k_base`:

* `"p50k_base"` – used by older models like `davinci`.
* `"r50k_base"` – used by `text-davinci-002`.
* `"gpt2"` – legacy models.

In [ ]:
import tiktoken


def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens = num_tokens_from_string(question, "cl100k_base")
num_tokens

### Text embedding models
- https://python.langchain.com/docs/integrations/text_embedding/openai/

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cuda"})  # Use HuggingFaceEmbeddings instead of OpenAIEmbeddings

query_result = embedding_model.embed_query(question)
document_result = embedding_model.embed_query(document)
len(query_result)

### Similarity:
* Cosine similarity is recommended (1 indicates identical) for OpenAI embeddings.

In [ ]:
import numpy as np


def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)


similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

---

### Document loaders
- DocumentLoaders load data into the standard LangChain Document format.
- Each DocumentLoader has its own specific parameters, but they can all be invoked in the same way with the .load method.

In [ ]:
#### INDEXING ####

# Load blog
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

print(f"Loaded {len(blog_docs)} documents from the blog.")
print(blog_docs[0].page_content[:500])  # Print the first 500 characters of the first document


### Spiltter:
- This text splitter is the recommended one for generic text. It is parameterized by a list of characters. It tries to split on them in order until the chunks are small enough. The default list is ["\n\n", "\n", " ", ""]. This has the effect of trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generically seem to be the strongest semantically related pieces of text.

In [ ]:
# Split
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)

# Make splits
splits = text_splitter.split_documents(blog_docs)
# splits[:2]  # Display the first two splits

The difference between of two ways to split documents using LangChain's `RecursiveCharacterTextSplitter`

---

### ✅ **Commonality Between Both**

Both versions:

* Use `RecursiveCharacterTextSplitter` to break long documents into smaller chunks.
* Accept parameters like `chunk_size` and `chunk_overlap`.
* Produce `splits`, which is a list of `Document` chunks.

---

### 🔍 **Key Difference: The `from_tiktoken_encoder` Method**

```python
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)
```

This version **uses a tokenizer-based approach** (specifically `cl100k_base`) to split **based on token count**, not just character length.

#### ▶️ Internally:

* Uses `tiktoken` (OpenAI’s tokenizer) to count tokens.
* `chunk_size=300` means 300 **tokens**, not characters.
* Better for **LLMs**, since tokens are what they actually care about (e.g., prompt length, embedding limits).

---

### 🧱 First Version: Character-based splitting

```python
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
```

* Splits based purely on **character count**, not tokens.
* Faster, but less accurate when dealing with models that care about **token limits**.
* `chunk_size=1000` = 1000 characters (not tokens!).

---

### ✅ Which One Should You Use?

| Use Case                            | Recommended Splitter                     |
| ----------------------------------- | ---------------------------------------- |
| Working with LLMs (GPT, embeddings) | `from_tiktoken_encoder` (token-based)    |
| Quick, rough character splitting    | Regular `RecursiveCharacterTextSplitter` |

---

### 📌 Summary

| Feature                | `RecursiveCharacterTextSplitter` | `from_tiktoken_encoder()` |
| ---------------------- | -------------------------------- | ------------------------- |
| Measures               | Characters                       | Tokens (`cl100k_base`)    |
| Better for LLMs        | ❌                                | ✅                         |
| Accuracy in token size | ❌ Approximate                    | ✅ Token-accurate          |
| Overlap units          | Characters                       | Tokens                    |

---


In [ ]:
# Index
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
# from langchain_community.embeddings import HuggingFaceEmbeddings  # Instead of OpenAIEmbeddings if you want to use HuggingFace
from langchain_huggingface import HuggingFaceEmbeddings  # ✅ Updated

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cuda"})  # Use HuggingFaceEmbeddings instead of OpenAIEmbeddings

vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)

# retriever = vectorstore.as_retriever()
retriever = vectorstore.as_retriever(search_kwargs={"k": 2, "score_threshold": 0.7})

In [ ]:
# Create retriever
retriever = vectorstore.as_retriever()
retriever.search_kwargs["k"] = 5

In [ ]:
# Retrieve relevant docs
results = retriever.invoke("What is this task about?")
print(results[0].page_content)  # ✅ Correct way to view the first result

> In the line:

```python
retriever.search_kwargs["k"] = 5
```

the parameter `k` refers to:

#### 🔍 **The number of top documents to retrieve**

When you query the retriever (e.g., `retriever.invoke("What is Task Decomposition?")`), it **searches the vectorstore** (like Chroma) and returns the **top `k` most similar documents** (based on vector similarity to your query).

---

#### 💡 Why it matters:

* **Higher `k`** means more documents are retrieved → potentially more context for the LLM.
* **Lower `k`** reduces noise but risks missing relevant information.

---

#### ✅ Example:

```python
retriever.search_kwargs["k"] = 3

results = retriever.invoke("What is RAG?")
for i, doc in enumerate(results):
    print(f"Doc {i+1}:")
    print(doc.page_content)
    print("------")
```

This retrieves **3 most relevant chunks** from your vector database.

---

#### 📌 Summary:

| Parameter | Meaning                                                 |
| --------- | ------------------------------------------------------- |
| `k`       | Number of top documents to retrieve                     |
| Default   | Usually `k=4` if not set (but varies)                   |
| Purpose   | Controls how many docs the retriever returns to the LLM |


### Vector stores
* A vector store stores embedded data and performs similarity search.
* The retriever is performing a KNN search in vector space to find the k most relevant document chunks based on semantic similarity.

### Part 3: Retrieval


In [ ]:
# Index
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings  # ✅ Updated
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cuda"})  # Use HuggingFaceEmbeddings
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)


retriever = vectorstore.as_retriever()


In [ ]:
results = retriever.invoke("What is Task Decomposition?",)  # ✅ Correct way to invoke the retriever
results

In [ ]:
# Make sure retriever is configured with k=3
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Correct usage with .invoke()
docs = retriever.invoke("What is Task Decomposition?")

In [ ]:
docs = retriever.get_relevant_documents("What is Task Decomposition?", k = 3)  # ✅ Correct way to get relevant documents
print(len(docs))  # Should print the number of relevant documents found
print("Content of the documents:\n")
print(docs[0].page_content.strip())  # ✅ Correct way to view the first


In [ ]:
print("Answer:\n")
print(results[0].page_content.strip())  # ✅ Correct way to view the first result

🔍 **What is Similarity Search?**

**Similarity Search** means:

> Given a query (like a question), find the most similar items (documents, paragraphs, etc.) from a database.

---

📦 **In LangChain (and RAG systems):**

When you use a **retriever** (like this):

```python
retriever = vectorstore.as_retriever()
retriever.search_kwargs["k"] = 5
```

You’re telling it:

> “Find the **top 5 documents** that are **most similar** to the user’s query.”

---

✅ **How Similarity Is Measured**

1. **Text is converted into vectors** using an **embedding model** (like Sentence Transformers or OpenAI Embeddings).
2. The system compares the **query vector** to all **document vectors** using **cosine similarity** (or Euclidean distance).
3. It returns the **k most similar** documents — just like a **KNN (K-Nearest Neighbors)** search.

---

**Example**:

You ask:

> "What is task decomposition?"

Your system:

1. Embeds that question into a vector.
2. Searches your **vectorstore** for the top `k=5` closest vectors (documents).
3. Returns those 5 documents as context for the LLM to answer your question.



### Connecting retrieval with an LLM via prompt
![Connecting retrieval with an LLM via prompt.png](<attachment:Connecting retrieval with an LLM via prompt.png>)

### Part 4: Generation wih Connecting retrieval with an LLM via prompt

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
    You are a helpful AI assistant. Use the following context to answer the question.

    Context:
    {context}

    Question: {question}

    Answer:"""

# prompt = ChatPromptTemplate.from_messages([
#     {"role": "user", "content": "What is task decomposition?"}
# ])

prompt = ChatPromptTemplate.from_template(prompt_template)
prompt

In [ ]:
# from langchain_core.prompts import PromptTemplate
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnablePassthrough

# prompt_template = ChatPromptTemplate.from_template(
# """
#     You are a helpful AI assistant. Use the following context to answer the question.

#     Context:
#     {context}

#     Question: {question}

#     Answer:"""
# )

# # prompt = ChatPromptTemplate.from_messages([
# #     {"role": "user", "content": "What is task decomposition?"}
# # ])

# prompt = ChatPromptTemplate.from_messages([
#     {"role": "user", "content": "What is task decomposition?"}
# ])
# prompt

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0.0,
    max_tokens=1024,
)

In [ ]:
rag_chain = (
    prompt
    | llm
    # | StrOutputParser(  # type: ignore
    #     output_key="answer",
    #     input_key="question",
    #     output_variable_name="answer",
    # )
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
context = format_docs(docs)
prompt_result = rag_chain.invoke({"context": context, "question": "What is task decomposition?"})
prompt_result

In [ ]:
from langchain import hub

prompt_hub_rag = hub.pull("rlm/rag-prompt")

In [ ]:
prompt_hub_rag

In [ ]:
# Prompt
prompt = hub.pull("rlm/rag-prompt")

# LLM
# llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
llm = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0.0,
    max_tokens=1024,
)


# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    # | StrOutputParser(  # type: ignore
    #     output_key="answer",
    #     input_key="question",
    #     output_variable_name="answer",
    # )
)

# Question
ans = rag_chain.invoke("What is Task Decomposition?")

> LANGCHAIN_API_KEY = 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

In [ ]:
# export LANGCHAIN_API_KEY="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"
# export LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
# export LANGCHAIN_PROJECT="mini-rag-app"


In [ ]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = "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"
os.environ['GROQ_API_KEY'] = "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"

In [ ]:
import os
import bs4

from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableMap
from langsmith import Client

# === 🧪 1. Setup LangSmith === #
os.environ["LANGCHAIN_API_KEY"] = "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"
os.environ["LANGCHAIN_PROJECT"] = "mini-rag-app"

# Optional LangSmith client (for test runs or metadata handling)
client = Client()

# === 📄 2. Load Documents === #
def load_docs():
    loader = WebBaseLoader(
        web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
        bs_kwargs=dict(
            parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))
        )
    )
    docs = loader.load()
    return docs

# Wrap with LangSmith config (optional)
def load_docs_with_trace():
    docs = load_docs()
    client.create_run(
        name="DocLoader",
        run_type="tool",  # ✅ FIXED
        inputs={"url": "https://lilianweng.github.io/posts/2023-06-23-agent/"},
        outputs={"num_docs": len(docs)},
        tags=["loader", "rag"],
        metadata={"stage": "load", "user": "Mohammed"}
    )
    return docs

# === ✂️ 3. Split Documents === #
def split_docs(docs):
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=300, chunk_overlap=50
    )
    splits = splitter.split_documents(docs)
    return splits

# Optional wrapper with tracing
def split_docs_with_trace(docs):
    splits = split_docs(docs)
    client.create_run(
        name="Splitter",
        run_type="tool",  # ✅ FIXED
        inputs={"num_input_docs": len(docs)},
        outputs={"num_chunks": len(splits)},
        tags=["splitter", "rag"],
        metadata={"stage": "split", "user": "Mohammed"}
    )
    return splits

# === 🧠 4. Embedding + Vector Store === #
def get_vectorstore(splits):
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
    )
    return Chroma.from_documents(documents=splits, embedding=embedding)

# === 🧾 5. Prompt + LLM === #
prompt_template = """
You are a helpful assistant. Use the context below to answer the question.

Context:
{context}

Question: {question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

llm = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0.0,
    max_tokens=1024,
)

# === 🔗 6. Chain Definition === #
rag_chain = RunnableMap({
    "context": lambda x: x["context"],
    "question": lambda x: x["question"],
}) | prompt | llm

# Wrap with LangSmith metadata
rag_chain = rag_chain.with_config(run_name="DeepSeek-RAG-Chain")

# === 🔁 7. Full RAG Pipeline === #
def run_rag(query: str) -> str:
    # Load and split with trace logging
    docs = load_docs_with_trace()
    splits = split_docs_with_trace(docs)

    # Vectorstore and retriever
    vectorstore = get_vectorstore(splits)
    retriever = vectorstore.as_retriever().with_config(run_name="RAG-Retriever")

    # Retrieve context
    relevant_docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # Invoke chain with LangSmith config
    response = rag_chain.invoke(
        {"context": context, "question": query},
        config={
            "run_name": "RAG Q&A",
            "tags": ["rag", "deepseek", "web-load", "langsmith"],
            "metadata": {
                "query_source": "manual",
                "user": "Mohammed",
                "document_source": "lilianweng/blog"
            },
        }
    )

    return response.content


In [ ]:
if __name__ == "__main__":
    answer = run_rag("What are the main components of an autonomous agent?")
    print("Answer:\n", answer)

## Query Transformations

![Rag From Scratch: Query Transformations.png](<attachment:Rag From Scratch: Query Transformations.png>)

## Part 5: Multi Query

![Multi Query.png](<attachment:Multi Query.png>)

**Multi-Query Retrieval (MQR)** in LangChain, especially in the context of Retrieval-Augmented Generation (RAG).

---

### 🧠 What Is Multi-Query Retrieval?

**Multi-Query Retrieval** is a technique where, instead of using a **single user query** to search for documents in a vector store, we generate **multiple diverse versions of that query**. Then we use each version to retrieve documents separately, and finally **combine all the results**.

##### ⚠️ Problem with Single Query Retrieval

A single query might:

* **Miss relevant documents** because it doesn’t match wording in the docs.
* Be **too specific or too vague**, depending on phrasing.
* Perform poorly in **semantic similarity-based** vector searches due to vocabulary mismatch.

---

#### ✅ Solution: Multi-Query Retrieval

**Steps:**
1. **User inputs a single query**:
   e.g., `"What is task decomposition for LLM agents?"`

2. **LLM generates diverse reformulations**:

   * `"How do LLMs break down tasks?"`
   * `"Explain how agents use task planning."`
   * `"What are the components of task decomposition?"`
   * etc.

3. **Each sub-query is run separately** on the vectorstore retriever.

4. **All retrieved results are merged**, and duplicates are removed.

5. The final context is used to **answer the original question** using the LLM.

---

#### 🧪 Why Multi-Query Works

* **Increases recall**: Finds more relevant passages across multiple phrasings.
* **Captures semantic variation**: Especially useful when queries and documents use different terminology.
* **Reduces hallucinations**: Because retrieval is richer and more grounded.

---

#### 🔁 Comparison

| Feature            | Single Query         | Multi-Query                                 |
| ------------------ | -------------------- | ------------------------------------------- |
| Retrieval coverage | Narrow               | Broad                                       |
| Recall             | Often lower          | Higher (more chances to find relevant docs) |
| Redundancy         | Low                  | Needs deduplication (which we handle)       |
| Accuracy           | Can miss key context | More robust and informative                 |


In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

# === Step 1: Load the blog content ===
def load_blog():
    loader = WebBaseLoader(
        web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
        bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header")))
    )
    return loader.load()

# === Step 2: Split into chunks ===
def split_documents(docs):
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=300, chunk_overlap=50)
    return splitter.split_documents(docs)

# === Step 3: Embed and Index ===
def create_vectorstore(splits):
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cuda"}  # Change to "cpu" if needed
    )
    return Chroma.from_documents(documents=splits, embedding=embedding_model)

# === Step 4: Define prompt for generating multiple queries ===
multi_query_prompt = ChatPromptTemplate.from_template("""You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""
)

# === Step 5: Multi-query generation chain ===
generate_queries_chain = (
    multi_query_prompt
    | ChatGroq(model="deepseek-r1-distill-llama-70b", temperature=0.0)
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
)

# === Step 6: Retrieve docs for each reformulated query ===
def retrieve_documents(generated_queries, retriever, top_k=3):
    results = {}
    for query in generated_queries:
        docs = retriever.invoke(query)[:top_k]
        results[query] = docs
    return results

In [ ]:
# === Step 7: Run Full Pipeline ===
def run_multi_query_rag(question: str):
    print(f"\n🔎 Original Question:\n{question}\n")

    # Load and prepare
    docs = load_blog()
    splits = split_documents(docs)
    vectorstore = create_vectorstore(splits)
    retriever = vectorstore.as_retriever()

    # Generate multiple queries
    generated_queries = generate_queries_chain.invoke({"question": question})
    print("🧠 Reformulated Queries:")
    for i, q in enumerate(generated_queries, 1):
        print(f"{i}. {q}")

    # Retrieve docs
    retrieved_docs = retrieve_documents(generated_queries, retriever, top_k=3)

    print("\n📄 Retrieved Chunks:")
    for query, docs in retrieved_docs.items():
        print(f"\n🔹 Query: {query}")
        for i, doc in enumerate(docs, 1):
            preview = doc.page_content[:200].replace("\n", " ") + "..."
            print(f"  {i}. {preview}")

    return {
        "original_question": question,
        "generated_queries": generated_queries,
        "retrieved_documents": retrieved_docs
    }

In [ ]:
# === Step 8: Run example ===
if __name__ == "__main__":
    question = "What is task decomposition for LLM agents?"
    run_multi_query_rag(question)


🔎 Original Question:
What is task decomposition for LLM agents?

🧠 Reformulated Queries:
1. <think>
2. Okay, so I need to figure out how to generate five different versions of the user's question about task decomposition for LLM agents. The goal is to help retrieve relevant documents from a vector database by providing alternative perspectives.
3. First, I should understand what task decomposition means in the context of LLM agents. Task decomposition is about breaking down complex tasks into simpler subtasks that the model can handle step by step. So, the original question is "What is task decomposition for LLM agents?"
4. Now, I need to think of different ways to phrase this question. Maybe start by rephrasing "task decomposition" with synonyms or related terms. For example, "breaking down tasks" or "subtask breakdown."
5. Another angle could be focusing on the purpose or benefit, like "importance of task decomposition" or "role of task decomposition."
6. I should also consider diff

In [ ]:
from langchain.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

In [ ]:
from operator import itemgetter
# from langchain_openai import ChatOpenAI
# from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0.0,
    max_tokens=1024,
)

final_rag_chain = (
    {"context": retrieval_chain,
     "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

## Part 6: RAG-Fusion

![RAG-Fusion.png](attachment:RAG-Fusion.png)

### 🧩 What Is RAG Fusion?

**RAG Fusion** is an **improved RAG approach** that combines the outputs of multiple retrievals to **generate a single, high-quality final answer**.

It was introduced in the paper:

> **"RAG-Fusion: Leveraging Multiple Retrievals for Knowledge-Intensive NLP Tasks"**
> (Lewis et al., 2020, Facebook AI)

#### 🧠 Goal:

Instead of relying on a **single set of documents** from one query, RAG Fusion **aggregates multiple retrieved contexts** (e.g., from multiple queries or reformulations), **ranks** them, and uses that **fused set** to guide the answer.

---

### 💡 Why Do We Need RAG Fusion?

* Different phrasings (like in Multi-Query) retrieve **different** relevant documents.
* Simply concatenating all docs can overwhelm the LLM (especially with token limits).
* So we need a way to:

  1. Merge retrieved docs.
  2. Rank by relevance.
  3. Select the **best subset** to pass to the LLM.

---

### 🛠️ RAG Fusion Pipeline Breakdown

#### Step-by-Step:

1. **Multiple Query Reformulations**
   → Like in multi-query RAG
   → e.g. 5 versions of a question

2. **Retrieve Top-k Docs for Each Subquery**
   → Use the vectorstore retriever

3. **Rank All Retrieved Docs Together**
   → Use a **cross-encoder** (optional) or similarity + heuristics
   → Score all docs across all queries

4. **Fuse Top-ranked Docs into a Final Set**
   → Deduplicate + sort by score
   → Select top-N docs to pass to LLM

5. **Generate Answer from Fused Context**
   → Prompt the LLM with final context chunk

---

* Docs:
  - https://github.com/langchain-ai/langchain/blob/master/cookbook/rag_fusion.ipynb?ref=blog.langchain.dev
* Blog / repo:
  - https://towardsdatascience.com/forget-rag-the-future-is-rag-fusion-1147298d8ad1


In [2]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = "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"
os.environ['GROQ_API_KEY'] = "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"

In [1]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load blog post
loader = WebBaseLoader(
    web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs={"parse_only": bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))},
)
docs = loader.load()

# 2. Split documents
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)
splits = text_splitter.split_documents(docs)

# 3. Embed and store in Chroma
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2", model_kwargs={"device": "cuda"}
)
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
retriever = vectorstore.as_retriever()


USER_AGENT environment variable not set, consider setting it to identify your requests.
/tmp/ipykernel_371915/374071122.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/home/developer/Dev/Machine-Learning-Projects/Build RAG App From Scratch/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-20 20:31:08.768940: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numeric

In [3]:
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LLM
llm = ChatGroq(
    model="deepseek-r1-distill-llama-70b",
    temperature=0.0,
    max_tokens=1024,
)

# Prompt to generate alternative queries
multi_query_prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries)"""
)

# Chain to generate multiple queries
generate_queries = multi_query_prompt | llm | StrOutputParser() | (lambda x: x.split("\n"))


In [4]:
from langchain.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = {}
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            fused_scores[doc_str] = fused_scores.get(doc_str, 0) + 1 / (rank + k)
    reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return [loads(doc) for doc, _ in reranked]


In [5]:
from operator import itemgetter

# Compose retrieval chain
retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion


In [6]:
# Final answer generation prompt
rag_prompt = ChatPromptTemplate.from_template(
    """Answer the following question based on the provided context:

{context}

Question: {question}
"""
)

# Full RAG Fusion chain
final_rag_chain = (
    {
        "context": retrieval_chain_rag_fusion,
        "question": itemgetter("question")
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)


In [7]:
question = "What is task decomposition for LLM agents?"

# Get final answer
answer = final_rag_chain.invoke({"question": question})
print(answer)


/tmp/ipykernel_371915/4172292832.py:10: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc, _ in reranked]


<think>
Okay, so I need to figure out what task decomposition is for LLM agents based on the provided context. Let me start by reading through the documents to understand the concept.

From the first document, I see that task decomposition is part of planning. It mentions that a complicated task usually involves many steps, and an agent needs to break it down into smaller, manageable parts. The document talks about Chain of Thought (CoT) and Tree of Thoughts (ToT) as prompting techniques. CoT makes the model think step by step, decomposing big tasks into smaller ones. ToT extends this by exploring multiple possibilities at each step, creating a tree structure with BFS or DFS.

Another document describes the ReAct prompt template, which includes steps like Thought, Action, Observation, and repeats them. This suggests that decomposition involves iterative steps where the model thinks, acts, and observes outcomes, refining its approach each time.

Looking further, there's a mention of how

* Trace:
    - https://smith.langchain.com/public/071202c9-9f4d-41b1-bf9d-86b7c5a7525b/r

Papers are used:

- https://arxiv.org/pdf/2205.10625.pdf
- https://arxiv.org/abs/2212.10509.pdf